# Heart Disease — Exploratory Data Analysis & Preprocessing

**Dataset:** UCI Heart Disease (918 records, 12 features)  
**Target:** `HeartDisease` (0 = No, 1 = Yes)  
**Flow:** Load → Inspect → Fix data quality issues → Visualize → Encode → Scale

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Load Data

In [ ]:
df = pd.read_csv('../datasets/heart.csv')
print(df.shape)
df.head(10)

In [ ]:
df.info()
print('\n')
df.describe()

In [ ]:
print('Nulls:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())

## 3. Univariate Analysis — Numeric Features

In [ ]:
numeric_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Data Quality — Biologically Impossible Zeros

Cholesterol and RestingBP cannot be 0 in a living person. These are recording errors — replaced with column mean (computed excluding zeros).

In [ ]:
print('Zero Cholesterol:', (df['Cholesterol'] == 0).sum())
print('Zero RestingBP: ', (df['RestingBP'] == 0).sum())

In [ ]:
for col in ['Cholesterol', 'RestingBP']:
    col_mean = df.loc[df[col] != 0, col].mean()
    df[col] = df[col].replace(0, col_mean).round(2)

print('Zeros remaining — Cholesterol:', (df['Cholesterol'] == 0).sum(),
      '| RestingBP:', (df['RestingBP'] == 0).sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} (after fix)')
plt.tight_layout()
plt.show()

## 5. Categorical Features vs Target

In [ ]:
cat_cols = ['Sex', 'ChestPainType', 'FastingBS']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, cat_cols):
    sns.countplot(x=df[col], hue=df['HeartDisease'], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x='HeartDisease', y='Cholesterol', data=df, ax=axes[0])
sns.violinplot(x='HeartDisease', y='Age', data=df, ax=axes[1])
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
corr = df.corr(numeric_only=True)
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nTop correlations with HeartDisease:')
print(corr['HeartDisease'].sort_values(ascending=False))

## 7. Encoding & Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

df_encoded = pd.get_dummies(df, drop_first=True).astype(int)

scaler = StandardScaler()
scale_cols = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']
df_encoded[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print('Final shape:', df_encoded.shape)
df_encoded.head()